In [ ]:
import os
import kagglehub
import shutil

path = kagglehub.competition_download('house-prices-advanced-regression-techniques')
print("Fichiers téléchargés dans :", path)

# Dossier de destination
dest = "data"
#os.makedirs(dest, exist_ok=True)

# Copier tous les fichiers du dossier téléchargé vers /data
#for fichier in os.listdir(path):
#    src_file = os.path.join(path, fichier)
#    dst_file = os.path.join(dest, fichier)
#    shutil.copy2(src_file, dst_file)

print("Fichiers copiés dans :", dest)

In [13]:
import pandas as pd
import matplotlib as plt
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import importlib
import preprocessing as p
importlib.reload(p)

df_train=pd.read_csv('data/train.csv')
df_test=pd.read_csv('data/test.csv')

In [9]:
X=df_train.drop(columns=['SalePrice'])
y=df_train['SalePrice']
X_train,X_valid,y_train,y_valid=train_test_split(X,y,test_size=0.2,random_state=42)

In [10]:
test_ids = df_test['Id'].copy()
X_train,power_transform,imputer=p.preprocessing(X_train)
X_valid,_,_=p.preprocessing(X_valid,power_transform,imputer)
df_test,_,_=p.preprocessing(df_test,power_transform,imputer)

In [11]:
model = LinearRegression()
model.fit(X_train, np.log1p(y_train))

y_pred = model.predict(X_valid)
rmse = np.sqrt(mean_squared_error(np.log1p(y_valid), y_pred))
print(rmse)

0.43325695782304796


In [12]:
erreurs = np.abs(y_pred - np.log1p(y_valid).values)
idx = np.argsort(erreurs)[-5:]   # les 5 pires
print("pred :", y_pred[idx])
print("vrai :", np.log1p(y_valid).values[idx])
print("écart:", erreurs[idx])

print(X_valid.describe().loc[['min','max']].T.sort_values('max').tail())
print(X_valid.describe().loc[['min','max']].T.sort_values('min').head())

print(X_train['Utilities'].max(), X_train['Functional'].max())

pred : [12.03082907 12.03082907 12.03082907 12.03082907 12.03082907]
vrai : [13.22956979 13.32392858 10.59665973 13.53447435 10.47197813]
écart: [1.19874072 1.29309952 1.43416933 1.50364529 1.55885094]
                     min           max
YrSold      2.006000e+03  2.010000e+03
BsmtUnfSF   0.000000e+00  2.042000e+03
PavedDrive  5.660015e+04  6.445289e+10
Functional  4.972361e+25  4.984177e+41
Utilities   4.978055e+51  4.978055e+51
              min        max
BsmtQual      0.0  17.212464
BsmtExposure  0.0   1.042523
BsmtFinType1  0.0   6.000000
MasVnrArea    0.0   3.706729
FullBath      0.0   3.000000
4.978055213497185e+51 4.984176749671765e+41
